# AutoClips AI — تجربة من الموبايل
ارفع فيديو بودكاست عربي أو إنجليزي، ثم شغّل الخلايا بالترتيب. Google Colab المجاني محدود الموارد وغير مضمون التوفر دائمًا.
**تنبيه:** لا ترفع فيديوهات خاصة أو حساسة؛ الملفات تُعالَج داخل جلسة Colab الخاصة بك.


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto-core > /dev/null
!pip -q install 'faster-whisper>=1.1,<2'
!git clone -q https://github.com/Y-fr3wn/autoclips-ai.git /content/autoclips-ai
!ffmpeg -hide_banner -filters | grep subtitles


## 1 — ارفع الفيديو
اضغط اختيار ملف وارفع MP4 أو MOV. ابدأ بفيديو قصير (2–5 دقائق) لاختبار النسخة الأولى.


In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('لم يتم اختيار فيديو')
video_path = str(Path('/content') / next(iter(uploaded)))
print('تم اختيار:', video_path)


## 2 — تقطيع تلقائي
المحرك يختار مقاطع وفق تفريغ الكلام والكلمات الجاذبة، وليس تحليلًا دلاليًا متقدمًا بعد. قد لا يجد مقطعًا مناسبًا في الفيديو القصير جدًا.


In [ ]:
import subprocess, sys
from pathlib import Path
out = '/content/autoclips-output'
cmd = [sys.executable, '/content/autoclips-ai/engine/clipper.py', video_path,
       '--output', out, '--count', '3', '--min-seconds', '15',
       '--max-seconds', '60', '--model', 'tiny', '--subtitles']
print('بدء التحليل والتقطيع...')
subprocess.run(cmd, check=True)
print('الملفات الناتجة:', list(Path(out).glob('*.mp4')))


## 3 — تنزيل الشورتس
لو اتولدت مقاطع، هتتنزل في ملف ZIP واحد.


In [ ]:
import shutil
from google.colab import files
from pathlib import Path
mp4s = list(Path('/content/autoclips-output').glob('*.mp4'))
if not mp4s:
    print('لم يتم توليد مقاطع. جرب فيديو أطول أو مدة أقل.')
else:
    archive = shutil.make_archive('/content/autoclips-shorts', 'zip', '/content/autoclips-output')
    files.download(archive)
